# Sparkle V2 — Fase 2+3 v2: Extracción extendida (MAR + blendshapes)

Extiende la extracción de la Fase 2+3 original. El baseline con solo EAR + yaw + pitch quedó **por debajo del clasificador trivial** (36.9% vs 45.6%), diagnosticado como falta de señal en las features, no problema de arquitectura (ver documento de avance metodológico).

**Features nuevas respecto de la v1:**
- **MAR (Mouth Aspect Ratio):** análogo al EAR pero para la boca — indicador directo de bostezo/somnolencia.
- **Blendshapes de MediaPipe (subconjunto curado, 15 de 52):**
  - Parpadeo: `eyeBlinkLeft`, `eyeBlinkRight` (refuerza el EAR)
  - Entrecerrar los ojos: `eyeSquintLeft`, `eyeSquintRight` (fatiga)
  - **Dirección de mirada real** (no solo orientación de cabeza): `eyeLookInLeft/Right`, `eyeLookOutLeft/Right`, `eyeLookUpLeft/Right`, `eyeLookDownLeft/Right` (8 señales) — esto es nuevo, antes no medíamos gaze, solo head pose.
  - Cejas: `browDownLeft`, `browDownRight`, `browInnerUp` (concentración/confusión)
  - Mandíbula: `jawOpen` (bostezo, complementa MAR)

Total: 5 features originales + 1 MAR + 15 blendshapes = **21 features por cuadro**.

Guarda todo en `features_v2/` en Drive, sin tocar la extracción anterior (`features/`), para poder comparar resultados antes/después.

In [1]:
!pip install -q mediapipe

from google.colab import drive
drive.mount('/content/drive')

import os, glob, cv2, math
import numpy as np
import pandas as pd
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision
from tqdm.notebook import tqdm

PROJECT_DIR = '/content/drive/MyDrive/SparkleV2'
RAW_DATASET_DIR = '/content/daisee_raw/DAiSEE/DataSet'
LABELS_DIR = f'{PROJECT_DIR}/labels'
FEATURES_DIR = f'{PROJECT_DIR}/features_v2'

for split in ['Train', 'Validation', 'Test']:
    os.makedirs(f'{FEATURES_DIR}/{split}', exist_ok=True)

print('Listo.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 12.2 MB/s eta 0:00:00
Mounted at /content/drive
Listo.


## 1. Modelo de FaceLandmarker (mismo que la v1)

In [3]:
MODEL_PATH = '/content/face_landmarker.task'
if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task

print('Modelo listo:', os.path.exists(MODEL_PATH), '-', os.path.getsize(MODEL_PATH), 'bytes')

Modelo listo: True - 3758596 bytes


## 2. Dataset raw y labels (si es sesión nueva, re-descargar)

In [5]:
if not os.path.exists(RAW_DATASET_DIR):
    print('No encontré el dataset en este runtime. Volviendo a descargar desde Kaggle...')
    from google.colab import files
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        print('Subí de nuevo tu kaggle.json:')
        uploaded = files.upload()
        os.makedirs('/root/.kaggle', exist_ok=True)
        !cp kaggle.json /root/.kaggle/kaggle.json
        !chmod 600 /root/.kaggle/kaggle.json
    os.makedirs('/content/daisee_raw', exist_ok=True)
    !kaggle datasets download -d olgaparfenova/daisee -p /content/daisee_raw --unzip
else:
    print('Dataset ya presente en este runtime, seguimos.')

Dataset ya presente en este runtime, seguimos.


In [6]:
def load_labels(split):
    df = pd.read_csv(f'{LABELS_DIR}/{split}Labels.csv')
    df.columns = [c.strip() for c in df.columns]
    return df

labels = {
    'Train': load_labels('Train'),
    'Validation': load_labels('Validation'),
    'Test': load_labels('Test'),
}

video_index = {}
for split in ['Train', 'Validation', 'Test']:
    split_dir = f'{RAW_DATASET_DIR}/{split}'
    paths = glob.glob(f'{split_dir}/**/*.avi', recursive=True)
    for p in paths:
        video_index[os.path.basename(p)] = p

print(f'Videos indexados: {len(video_index)}')
for split, df in labels.items():
    print(f'{split}: {len(df)} labels')

Videos indexados: 3378
Train: 5358 labels
Validation: 1429 labels
Test: 1784 labels


## 3. Funciones de extracción de features (extendidas)

In [7]:
# Landmarks para EAR (igual que v1)
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
LEFT_EYE  = [362, 385, 387, 263, 373, 380]

# Landmarks para MAR (nuevo): comisuras (61, 291) y centro labio sup/inf (13, 14)
MOUTH_CORNERS = [61, 291]
MOUTH_VERTICAL = [13, 14]

# Subconjunto curado de blendshapes (de los 52 disponibles)
BLENDSHAPES_OF_INTEREST = [
    'eyeBlinkLeft', 'eyeBlinkRight',
    'eyeSquintLeft', 'eyeSquintRight',
    'eyeLookInLeft', 'eyeLookOutLeft', 'eyeLookUpLeft', 'eyeLookDownLeft',
    'eyeLookInRight', 'eyeLookOutRight', 'eyeLookUpRight', 'eyeLookDownRight',
    'browDownLeft', 'browDownRight', 'browInnerUp',
    'jawOpen',
]

FEATURE_NAMES = ['ear_avg', 'ear_left', 'ear_right', 'yaw', 'pitch', 'mar'] + BLENDSHAPES_OF_INTEREST
N_FRAMES_PER_CLIP = 60

print(f'Total de features por cuadro: {len(FEATURE_NAMES)}')
print(FEATURE_NAMES)

Total de features por cuadro: 22
['ear_avg', 'ear_left', 'ear_right', 'yaw', 'pitch', 'mar', 'eyeBlinkLeft', 'eyeBlinkRight', 'eyeSquintLeft', 'eyeSquintRight', 'eyeLookInLeft', 'eyeLookOutLeft', 'eyeLookUpLeft', 'eyeLookDownLeft', 'eyeLookInRight', 'eyeLookOutRight', 'eyeLookUpRight', 'eyeLookDownRight', 'browDownLeft', 'browDownRight', 'browInnerUp', 'jawOpen']


In [8]:
def create_face_landmarker():
    base_options = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
    options = mp_vision.FaceLandmarkerOptions(
        base_options=base_options,
        running_mode=mp_vision.RunningMode.IMAGE,
        num_faces=1,
        min_face_detection_confidence=0.5,
        output_facial_transformation_matrixes=True,
        output_face_blendshapes=True,   # NUEVO respecto de la v1
    )
    return mp_vision.FaceLandmarker.create_from_options(options)


def eye_aspect_ratio(landmarks, eye_ids, w, h):
    pts = np.array([(landmarks[i].x * w, landmarks[i].y * h) for i in eye_ids])
    vert1 = np.linalg.norm(pts[1] - pts[5])
    vert2 = np.linalg.norm(pts[2] - pts[4])
    horiz = np.linalg.norm(pts[0] - pts[3])
    if horiz == 0:
        return np.nan
    return (vert1 + vert2) / (2.0 * horiz)


def mouth_aspect_ratio(landmarks, w, h):
    left, right = MOUTH_CORNERS
    top, bottom = MOUTH_VERTICAL
    p_left = np.array([landmarks[left].x * w, landmarks[left].y * h])
    p_right = np.array([landmarks[right].x * w, landmarks[right].y * h])
    p_top = np.array([landmarks[top].x * w, landmarks[top].y * h])
    p_bottom = np.array([landmarks[bottom].x * w, landmarks[bottom].y * h])

    horiz = np.linalg.norm(p_left - p_right)
    if horiz == 0:
        return np.nan
    vert = np.linalg.norm(p_top - p_bottom)
    return vert / horiz


def euler_from_transformation_matrix(matrix):
    rot = np.array(matrix)[:3, :3]
    sy = math.sqrt(rot[0, 0] ** 2 + rot[1, 0] ** 2)
    singular = sy < 1e-6
    if not singular:
        pitch = math.degrees(math.atan2(rot[2, 1], rot[2, 2]))
        yaw = math.degrees(math.atan2(-rot[2, 0], sy))
    else:
        pitch = math.degrees(math.atan2(-rot[1, 2], rot[1, 1]))
        yaw = math.degrees(math.atan2(-rot[2, 0], sy))
    return yaw, pitch


def extract_features_from_frame(detector, frame_bgr):
    h, w = frame_bgr.shape[:2]
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)
    result = detector.detect(mp_image)

    n_total = len(FEATURE_NAMES)

    if not result.face_landmarks:
        return np.array([np.nan] * n_total)

    landmarks = result.face_landmarks[0]
    ear_r = eye_aspect_ratio(landmarks, RIGHT_EYE, w, h)
    ear_l = eye_aspect_ratio(landmarks, LEFT_EYE, w, h)
    ear_avg = (ear_r + ear_l) / 2.0
    mar = mouth_aspect_ratio(landmarks, w, h)

    yaw, pitch = np.nan, np.nan
    if result.facial_transformation_matrixes:
        yaw, pitch = euler_from_transformation_matrix(result.facial_transformation_matrixes[0])

    blendshape_values = {b.category_name: b.score for b in result.face_blendshapes[0]} if result.face_blendshapes else {}
    blendshape_vector = [blendshape_values.get(name, np.nan) for name in BLENDSHAPES_OF_INTEREST]

    return np.array([ear_avg, ear_l, ear_r, yaw, pitch, mar] + blendshape_vector, dtype=np.float64)

## 4. Función de procesamiento por clip (igual estructura que v1)

In [9]:
def process_clip(video_path, detector, n_frames=N_FRAMES_PER_CLIP):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames <= 0:
        cap.release()
        return None

    frame_indices = np.linspace(0, total_frames - 1, n_frames).astype(int)
    sequence = np.full((n_frames, len(FEATURE_NAMES)), np.nan)

    for seq_pos, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(frame_idx))
        ret, frame = cap.read()
        if not ret:
            continue
        sequence[seq_pos] = extract_features_from_frame(detector, frame)

    cap.release()
    return sequence

## 5. Prueba en un subset chico

Verificá que los blendshapes se estén extrayendo bien (no todo NaN) antes de lanzar el batch completo.

In [10]:
import time

test_clips = labels['Train'].head(10)
detector = create_face_landmarker()

try:
    t0 = time.time()
    for _, row in test_clips.iterrows():
        clip_id = row['ClipID']
        if clip_id not in video_index:
            continue
        seq = process_clip(video_index[clip_id], detector)
        pct_nan = np.isnan(seq).any(axis=1).mean() * 100
        print(f'{clip_id}: {pct_nan:.1f}% NaN | jawOpen medio: {np.nanmean(seq[:, FEATURE_NAMES.index("jawOpen")]):.4f} | '
              f'eyeBlinkLeft medio: {np.nanmean(seq[:, FEATURE_NAMES.index("eyeBlinkLeft")]):.4f}')
    elapsed = time.time() - t0
finally:
    detector.close()

print(f'\nTiempo: {elapsed:.1f}s para 10 clips ({elapsed/10:.2f}s/clip)')
print(f'Estimado dataset completo (~7900 clips): {elapsed/10*7900/3600:.1f} horas')

1100011002.avi: 0.0% NaN | jawOpen medio: 0.0063 | eyeBlinkLeft medio: 0.1226
1100011003.avi: 0.0% NaN | jawOpen medio: 0.0062 | eyeBlinkLeft medio: 0.0939
1100011004.avi: 0.0% NaN | jawOpen medio: 0.0040 | eyeBlinkLeft medio: 0.1009
1100011005.avi: 0.0% NaN | jawOpen medio: 0.0167 | eyeBlinkLeft medio: 0.0959
1100011006.avi: 0.0% NaN | jawOpen medio: 0.0055 | eyeBlinkLeft medio: 0.1290
1100011007.avi: 0.0% NaN | jawOpen medio: 0.0112 | eyeBlinkLeft medio: 0.1247
1100011008.avi: 0.0% NaN | jawOpen medio: 0.0428 | eyeBlinkLeft medio: 0.1283
1100011009.avi: 0.0% NaN | jawOpen medio: 0.0606 | eyeBlinkLeft medio: 0.0980
1100011010.avi: 0.0% NaN | jawOpen medio: 0.0329 | eyeBlinkLeft medio: 0.1154
1100011011.avi: 0.0% NaN | jawOpen medio: 0.0048 | eyeBlinkLeft medio: 0.1209

Tiempo: 24.5s para 10 clips (2.45s/clip)
Estimado dataset completo (~7900 clips): 5.4 horas


Si los valores de `jawOpen` y `eyeBlinkLeft` no son todos 0 o todos iguales, los blendshapes están funcionando. Pegame este output antes de lanzar el batch completo.

## 6. Batch completo con checkpointing (misma lógica que v1)

In [11]:
SPLIT = 'Train'  # cambiar a 'Validation' o 'Test' cuando corresponda

df_split = labels[SPLIT]
out_dir = f'{FEATURES_DIR}/{SPLIT}'
log_path = f'{FEATURES_DIR}/{SPLIT}_processing_log.csv'

log_rows = []
if os.path.exists(log_path):
    log_rows = pd.read_csv(log_path).to_dict('records')

already_done = set(os.path.splitext(f)[0] for f in os.listdir(out_dir) if f.endswith('.npz'))
print(f'{SPLIT}: {len(already_done)} clips ya procesados de {len(df_split)} totales')

Train: 0 clips ya procesados de 5358 totales


In [12]:
detector = create_face_landmarker()

try:
    for _, row in tqdm(df_split.iterrows(), total=len(df_split)):
        clip_id = row['ClipID']
        clip_key = os.path.splitext(clip_id)[0]

        if clip_key in already_done:
            continue
        if clip_id not in video_index:
            log_rows.append({'ClipID': clip_id, 'status': 'sin_video_fisico', 'pct_nan': None})
            continue

        try:
            seq = process_clip(video_index[clip_id], detector)
            if seq is None:
                log_rows.append({'ClipID': clip_id, 'status': 'video_ilegible', 'pct_nan': None})
                continue

            pct_nan = np.isnan(seq).any(axis=1).mean() * 100

            np.savez_compressed(
                f'{out_dir}/{clip_key}.npz',
                sequence=seq,
                feature_names=FEATURE_NAMES,
                boredom=row['Boredom'],
                engagement=row['Engagement'],
                confusion=row['Confusion'],
                frustration=row['Frustration'],
            )
            log_rows.append({'ClipID': clip_id, 'status': 'ok', 'pct_nan': pct_nan})

        except Exception as e:
            log_rows.append({'ClipID': clip_id, 'status': f'error: {e}', 'pct_nan': None})

        if len(log_rows) % 50 == 0:
            pd.DataFrame(log_rows).to_csv(log_path, index=False)
finally:
    detector.close()

pd.DataFrame(log_rows).to_csv(log_path, index=False)
print('Terminado (o cortado). Progreso guardado en', log_path)

  0%|          | 0/5358 [00:00<?, ?it/s]

Terminado (o cortado). Progreso guardado en /content/drive/MyDrive/SparkleV2/features_v2/Train_processing_log.csv


In [13]:
log_df = pd.read_csv(log_path)
print(log_df['status'].value_counts())
ok_df = log_df[log_df['status'] == 'ok']
if len(ok_df):
    print(f"\n% NaN promedio entre clips OK: {ok_df['pct_nan'].mean():.2f}%")
    print(f"Clips con >30% frames sin rostro: {(ok_df['pct_nan'] > 30).sum()}")

status
sin_video_fisico    3700
ok                  1657
video_ilegible         1
Name: count, dtype: int64

% NaN promedio entre clips OK: 0.00%
Clips con >30% frames sin rostro: 0


## Checkpoint de la Fase 2+3 v2

1. Corré primero la **sección 5** (subset de 10 clips) y pegame el output — quiero confirmar que los blendshapes tienen valores no triviales antes de lanzar el batch completo (que va a tardar un poco más que la v1 por el trabajo extra, aunque no debería ser mucho más dado que blendshapes se calcula en la misma llamada a `detect()`).
2. Repetí la sección 6 para los tres splits (`Train`, `Validation`, `Test`), igual que hicimos con la v1.

Cuando tengas los tres splits, seguimos con una Fase 4 v2 (tratamiento de faltantes) y Fase 5/6 v2 (reentrenar LSTM con las 21 features) para comparar contra el baseline documentado.

In [14]:
import glob
n_videos = len(glob.glob('/content/daisee_raw/DAiSEE/DataSet/**/*.avi', recursive=True))
print(f'Videos físicos encontrados: {n_videos}')

Videos físicos encontrados: 3378


In [15]:
import shutil
shutil.rmtree('/content/daisee_raw', ignore_errors=True)

os.makedirs('/content/daisee_raw', exist_ok=True)
!kaggle datasets download -d olgaparfenova/daisee -p /content/daisee_raw --unzip

Dataset URL: https://www.kaggle.com/datasets/olgaparfenova/daisee
License(s): unknown
100% 14.3G/14.3G [02:11<00:00, 117MB/s]



In [16]:
n_videos = len(glob.glob('/content/daisee_raw/DAiSEE/DataSet/**/*.avi', recursive=True))
print(f'Videos físicos encontrados: {n_videos}')

Videos físicos encontrados: 8232


In [17]:
for split in ['Train', 'Validation', 'Test']:
    n = len(glob.glob(f'/content/daisee_raw/DAiSEE/DataSet/{split}/**/*.avi', recursive=True))
    print(f'{split}: {n} videos físicos')

# Chequear que no se haya quedado sin espacio en disco durante la descarga/descompresión
!df -h /content

Train: 4976 videos físicos
Validation: 1536 videos físicos
Test: 1720 videos físicos
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   35G   74G  33% /
